# PathSnap — GPS / Road-Network Preprocessing

This notebook prepares candidate-road features for the PathSnap ML and HMM stages.

**Input:** the supplied ZIP archive containing `train/` and `student_test/` trip folders. Each trip folder is expected to contain `.track`, `.nodes`, `.arcs`, and (for training trips) `.route` files.

**Important:** only `train/` trips are labelled from `.route` and written to the training CSV. `student_test/` trips are processed without reading their `.route` files, so test labels are not used during preprocessing. This ZIP contains 80 `train` trips and 10 `student_test` trips; no evaluator partition was present in the uploaded archive.

Run the single code cell below in Google Colab. It asks you to upload the ZIP, processes all trips, and downloads a ZIP containing the CSV outputs.


In [ ]:

# ========================= PATHSNAP PREPROCESSING =========================
# Run this entire cell in Google Colab.
# It creates:
#   pathsnap_train_candidates.csv
#   pathsnap_student_test_candidates.csv
#   pathsnap_feature_columns.json
#   pathsnap_preprocessing_summary.csv
#   pathsnap_preprocessed_outputs.zip

import os, re, zipfile, tempfile, shutil, json, math, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from google.colab import files

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ----------------------------- SETTINGS ----------------------------------
K_NEAREST_ARCS = 15       # number of nearby road arcs considered per GPS point
EARTH_RADIUS_M = 6371000.0
OUTPUT_DIR = "/content/pathsnap_preprocessed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FEATURE_COLUMNS = [
    "distance_m",
    "heading_diff_deg",
    "speed_mps",
    "relative_distance_m",
    "connectivity"
]

# -------------------------- UPLOAD ZIP FILE -------------------------------
print("Upload your PathSnap dataset ZIP file...")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No ZIP file was uploaded.")
zip_name = next(iter(uploaded.keys()))
zip_path = f"/content/{zip_name}"
with open(zip_path, "wb") as f:
    f.write(uploaded[zip_name])

extract_dir = "/content/pathsnap_dataset"
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

# Find the dataset root, even if the ZIP has an enclosing folder.
all_track_files = list(Path(extract_dir).rglob("*.track"))
if not all_track_files:
    raise FileNotFoundError("No .track files found in the uploaded ZIP.")

def partition_for(track_path):
    parts = track_path.parts
    for part in parts:
        if part in ("train", "student_test", "evaluator_test"):
            return part
    return "unknown"

# ---------------------------- FILE READERS -------------------------------
def read_numeric_file(path, expected_cols):
    """Read a whitespace/tab-separated numeric file with no header."""
    arr = np.loadtxt(path, dtype=float, ndmin=2)
    if arr.shape[1] != expected_cols:
        raise ValueError(f"{path}: expected {expected_cols} columns, found {arr.shape[1]}")
    return arr

def read_route(path, number_of_arcs):
    """Route values are zero-based row indices into the .arcs file."""
    if not os.path.exists(path):
        return np.array([], dtype=int)
    vals = np.loadtxt(path, dtype=int, ndmin=1)
    vals = np.asarray(vals, dtype=int).reshape(-1)
    return vals[(vals >= 0) & (vals < number_of_arcs)]

# ------------------------- GEOMETRY HELPERS ------------------------------
def local_xy(lon, lat, lon0, lat0):
    """Local equirectangular projection in metres, suitable for one trip area."""
    x = EARTH_RADIUS_M * np.radians(lon - lon0) * np.cos(np.radians(lat0))
    y = EARTH_RADIUS_M * np.radians(lat - lat0)
    return np.column_stack([x, y])

def point_to_segments(points_xy, seg_a, seg_b):
    """Vectorised point-to-segment distances for one point vs many segments."""
    v = seg_b - seg_a
    denom = np.einsum("ij,ij->i", v, v)
    denom = np.where(denom == 0, 1e-12, denom)
    w = points_xy[None, :] - seg_a
    t = np.einsum("ij,ij->i", w, v) / denom
    t = np.clip(t, 0.0, 1.0)
    nearest = seg_a + t[:, None] * v
    return np.sqrt(np.einsum("ij,ij->i", points_xy[None, :] - nearest,
                             points_xy[None, :] - nearest))

def bearing_deg(a, b):
    """Bearing clockwise from north, using local x=east, y=north."""
    dx, dy = b[0] - a[0], b[1] - a[1]
    return (np.degrees(np.arctan2(dx, dy)) + 360.0) % 360.0

def angle_difference(a, b):
    return abs((a - b + 180.0) % 360.0 - 180.0)

def observation_motion(track_xy, timestamps, i):
    """Estimate heading and speed from neighbouring GPS observations."""
    n = len(track_xy)
    if n <= 1:
        return np.nan, 0.0
    left = max(0, i - 1)
    right = min(n - 1, i + 1)
    dt = float(timestamps[right] - timestamps[left])
    displacement = float(np.linalg.norm(track_xy[right] - track_xy[left]))
    speed = displacement / dt if dt > 0 else np.nan
    if displacement < 1e-6:
        heading = np.nan
    else:
        heading = bearing_deg(track_xy[left], track_xy[right])
    return heading, speed

# ------------------------- PROCESS ONE TRIP -------------------------------
def process_trip(trip_dir, partition):
    trip_dir = Path(trip_dir)
    trip_id = trip_dir.name

    track_path = trip_dir / f"{trip_id}.track"
    nodes_path = trip_dir / f"{trip_id}.nodes"
    arcs_path = trip_dir / f"{trip_id}.arcs"
    route_path = trip_dir / f"{trip_id}.route"

    track = read_numeric_file(track_path, 3)  # longitude, latitude, timestamp
    nodes = read_numeric_file(nodes_path, 2)  # longitude, latitude
    arcs = read_numeric_file(arcs_path, 2).astype(int)  # from_node, to_node

    if len(track) == 0 or len(nodes) == 0 or len(arcs) == 0:
        return pd.DataFrame(), {"trip_id": trip_id, "partition": partition,
                                "observations": len(track), "arcs": len(arcs),
                                "candidate_rows": 0, "status": "empty input"}

    # Validate node references and preserve .arcs row index as arc_id.
    valid_arc_mask = (
        (arcs[:, 0] >= 0) & (arcs[:, 0] < len(nodes)) &
        (arcs[:, 1] >= 0) & (arcs[:, 1] < len(nodes))
    )
    valid_arc_ids = np.flatnonzero(valid_arc_mask)
    arcs_valid = arcs[valid_arc_mask]
    if len(arcs_valid) == 0:
        return pd.DataFrame(), {"trip_id": trip_id, "partition": partition,
                                "observations": len(track), "arcs": len(arcs),
                                "candidate_rows": 0, "status": "no valid arcs"}

    # One local metric coordinate system per trip.
    lon0, lat0 = float(np.mean(track[:, 0])), float(np.mean(track[:, 1]))
    nodes_xy = local_xy(nodes[:, 0], nodes[:, 1], lon0, lat0)
    track_xy = local_xy(track[:, 0], track[:, 1], lon0, lat0)

    arc_a = nodes_xy[arcs_valid[:, 0]]
    arc_b = nodes_xy[arcs_valid[:, 1]]
    midpoints = (arc_a + arc_b) / 2.0
    midpoint_tree = cKDTree(midpoints)

    # Arc IDs remain the original zero-based row indices in the .arcs file.
    route_arc_ids = np.array([], dtype=int)
    route_tree = None
    route_midpoints = None
    route_arc_ids_valid = np.array([], dtype=int)

    # Only training trips use .route to create labels.
    if partition == "train" and route_path.exists():
        route_arc_ids = read_route(route_path, len(arcs))
        route_arc_ids_valid = route_arc_ids[np.isin(route_arc_ids, valid_arc_ids)]
        if len(route_arc_ids_valid):
            route_positions = np.searchsorted(valid_arc_ids, route_arc_ids_valid)
            route_midpoints = midpoints[route_positions]
            route_tree = cKDTree(route_midpoints)

    timestamps = track[:, 2].astype(float)
    rows = []
    previous_candidate_endpoints = set()

    for obs_i, point_xy in enumerate(track_xy):
        heading, speed = observation_motion(track_xy, timestamps, obs_i)

        k = min(K_NEAREST_ARCS, len(arcs_valid))
        _, nearest_local = midpoint_tree.query(point_xy, k=k)
        nearest_local = np.atleast_1d(nearest_local).astype(int).tolist()

        # For labelled training observations, add the nearest reference-route
        # arc even if midpoint search did not include it.
        positive_arc_id = None
        if route_tree is not None:
            _, route_pos = route_tree.query(point_xy, k=1)
            route_pos = int(route_pos)
            positive_arc_id = int(route_arc_ids_valid[route_pos])
            positive_local = int(np.searchsorted(valid_arc_ids, positive_arc_id))
            nearest_local.append(positive_local)

        nearest_local = list(dict.fromkeys(nearest_local))
        candidate_arc_ids = valid_arc_ids[np.asarray(nearest_local, dtype=int)]
        distances = point_to_segments(point_xy, arc_a[nearest_local], arc_b[nearest_local])
        min_dist = float(np.min(distances))

        current_endpoints = set()
        candidate_records = []
        for local_idx, arc_id, dist in zip(nearest_local, candidate_arc_ids, distances):
            from_node, to_node = arcs[int(arc_id)]
            current_endpoints.update((int(from_node), int(to_node)))

            seg_heading = bearing_deg(arc_a[local_idx], arc_b[local_idx])
            heading_diff = angle_difference(heading, seg_heading) if np.isfinite(heading) else np.nan

            # A simple topological continuity indicator: candidate shares
            # an endpoint with at least one candidate from previous observation.
            connected = int(
                bool(previous_candidate_endpoints) and
                (int(from_node) in previous_candidate_endpoints or
                 int(to_node) in previous_candidate_endpoints)
            )

            label = int(positive_arc_id is not None and int(arc_id) == positive_arc_id)
            candidate_records.append({
                "trip_id": str(trip_id),
                "observation_id": int(obs_i),
                "timestamp": float(timestamps[obs_i]),
                "arc_id": int(arc_id),
                "distance_m": float(dist),
                "heading_diff_deg": float(heading_diff) if np.isfinite(heading_diff) else np.nan,
                "speed_mps": float(speed) if np.isfinite(speed) else np.nan,
                "relative_distance_m": float(dist - min_dist),
                "connectivity": int(connected),
                "label": label if partition == "train" else np.nan,
                "partition": partition
            })

        rows.extend(candidate_records)
        previous_candidate_endpoints = current_endpoints

    result = pd.DataFrame(rows)
    summary = {
        "trip_id": trip_id,
        "partition": partition,
        "observations": int(len(track)),
        "nodes": int(len(nodes)),
        "arcs": int(len(arcs)),
        "valid_arcs": int(len(arcs_valid)),
        "candidate_rows": int(len(result)),
        "labelled_observations": int(result.groupby("observation_id")["label"].max().notna().sum())
            if partition == "train" and len(result) else 0,
        "positive_candidate_rows": int(result["label"].fillna(0).sum()) if len(result) else 0,
        "status": "ok"
    }
    return result, summary

# ---------------------------- RUN ALL TRIPS -------------------------------
all_frames = []
summaries = []
for track_file in sorted(all_track_files):
    partition = partition_for(track_file)
    if partition not in ("train", "student_test"):
        # Evaluator data, if ever present in another ZIP, is deliberately skipped.
        print(f"Skipping unsupported/protected partition: {track_file}")
        continue
    try:
        frame, summary = process_trip(track_file.parent, partition)
        if not frame.empty:
            all_frames.append(frame)
        summaries.append(summary)
        print(f"{partition:12s} {summary['trip_id']} | "
              f"obs={summary['observations']} | candidates={summary['candidate_rows']}")
    except Exception as e:
        summaries.append({"trip_id": track_file.parent.name, "partition": partition,
                          "status": f"ERROR: {type(e).__name__}: {e}"})
        print(f"ERROR processing {track_file.parent.name}: {e}")

if not all_frames:
    raise RuntimeError("No candidate rows were produced. Check the ZIP folder structure.")

combined = pd.concat(all_frames, ignore_index=True)
train_df = combined[combined["partition"] == "train"].copy()
student_test_df = combined[combined["partition"] == "student_test"].copy()

# Ensure the labelled training observations have one positive candidate each.
if not train_df.empty:
    positives_per_observation = train_df.groupby(["trip_id", "observation_id"])["label"].sum()
    print("\\nTraining observations with exactly one positive candidate:",
          int((positives_per_observation == 1).sum()), "/", len(positives_per_observation))
    print("Training observations with no positive candidate:",
          int((positives_per_observation == 0).sum()))

# Save outputs. Metadata columns must NOT be used as model features.
train_csv = os.path.join(OUTPUT_DIR, "pathsnap_train_candidates.csv")
test_csv = os.path.join(OUTPUT_DIR, "pathsnap_student_test_candidates.csv")
summary_csv = os.path.join(OUTPUT_DIR, "pathsnap_preprocessing_summary.csv")
feature_json = os.path.join(OUTPUT_DIR, "pathsnap_feature_columns.json")

train_df.to_csv(train_csv, index=False)
student_test_df.to_csv(test_csv, index=False)
pd.DataFrame(summaries).to_csv(summary_csv, index=False)
with open(feature_json, "w") as f:
    json.dump({"feature_columns": FEATURE_COLUMNS,
               "metadata_columns": ["trip_id", "observation_id", "timestamp", "arc_id"],
               "label_column": "label"}, f, indent=2)

# Zip all deliverables for one-click download.
output_zip = "/content/pathsnap_preprocessed_outputs.zip"
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in [train_csv, test_csv, summary_csv, feature_json]:
        zf.write(file_path, arcname=os.path.basename(file_path))

print("\\nPREPROCESSING FINISHED")
print("Training candidate rows:", len(train_df))
print("Student-test candidate rows:", len(student_test_df))
print("Feature columns:", FEATURE_COLUMNS)
print("\\nImportant integration notes:")
print("- arc_id is the zero-based row index in that trip's original .arcs file.")
print("- Every candidate row is retained; no probability threshold is applied here.")
print("- Use only FEATURE_COLUMNS as ML inputs. Keep trip_id/observation_id/timestamp/arc_id as metadata.")
print("- The student_test CSV has blank labels. Its .route files were not read.")
print("- This is candidate generation using nearest arc midpoints followed by exact point-to-segment distance.")
print("- The connectivity feature is a simple shared-endpoint indicator against candidates at the previous observation.")
print("- No road-speed attribute exists in these files, so road_speed_diff is intentionally not fabricated.")

files.download(output_zip)
# ==========================================================================
